In [21]:
import torch
from torchinfo import summary
from torch.utils.data import DataLoader, random_split
import pandas as pd
import os
from dotenv import load_dotenv
import lightning as L
from lightning.pytorch.callbacks import EarlyStopping
import sys
from tqdm.notebook import tqdm
import pickle

from torch.nn.utils.rnn import pad_sequence
from sklearn.metrics import accuracy_score, f1_score, classification_report

from src.transformer.transformer_architecture import KeypointTransformer, LitKeypointTransformer
from src.image_processing.mediapipe import MediaPipeProcessor
from src.datasets.ipn_data import IPNData

In [22]:
load_dotenv()
GESTURE_VIDEOS_PATH = os.getenv("IPN_GESTURE_VIDEOS")

In [23]:
# Dataset
df = pd.read_csv(f"{GESTURE_VIDEOS_PATH}/metadata.csv")
df.head()

,Video Name,Frames,Sex,Hand,Background,Illumination,People in Scene,Background Motion,Set,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17
0,1CM1_4_R__229,3751,W,Right,Clutter,Stable,Single,Static,train,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1CM1_4_R__230,3684,W,Right,Clutter,Stable,Single,Static,train,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1CM1_4_R__231,3747,W,Right,Clutter,Stable,Single,Static,train,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1CM1_4_R__232,3858,W,Right,Clutter,Stable,Single,Static,train,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1CM42_11_R__205,3686,M,Right,Plain,Stable,Single,Static,train,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
ds_raw = IPNData(GESTURE_VIDEOS_PATH)
mpp = MediaPipeProcessor()

W0000 00:00:1774301356.574716   19300 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1774301356.583100   19309 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [25]:
label_id_dict = ds_raw.get_label_id_dict()
num_videos = ds_raw.get_num_videos()
video_info_dict = ds_raw.get_video_info()

In [26]:
print(len(label_id_dict))
print(len(video_info_dict))

14
5649


In [28]:
label_to_index = {}

for k in label_id_dict:
    label, gesture = label_id_dict[k]
    label_to_index[label] = int(k)

In [ ]:
# Unnecessary if you have already processed dataset

N_V = num_videos
ds = []

for i in tqdm(range(N_V)):
    for j in range(sys.maxsize):
        try:
            frames, label, ID = ds_raw.__getitem__(i, j)

            keypoint_sequence = mpp.process_video(frames)

            ds.append([keypoint_sequence, label_to_index[label]]) # using ID as label
        except:
            break

In [9]:
#Saving new dataset

with open("data/dataset.pkl", "wb") as f:
    pickle.dump(ds, f)

print("Saved!")

Saved!


In [29]:
with open("data/dataset.pkl", "rb") as f:
    ds = pickle.load(f)

In [30]:
N_F = len(video_info_dict)
TEST_F = int(0.1*N_F)
TRAIN_VAL_F = N_F - TEST_F

train_size = int(0.9 * TRAIN_VAL_F)
val_size = TRAIN_VAL_F - train_size

test_ds = ds[TRAIN_VAL_F:]
train_val_ds = ds[:TRAIN_VAL_F]

train_ds, val_ds = random_split(train_val_ds, [train_size, val_size])

In [31]:
def collate_fn(batch):
    sequences, labels = zip(*batch)

    sequences = [torch.tensor(s, dtype=torch.float32) for s in sequences]

    # Padding
    padded_sequences = pad_sequence(sequences, batch_first=True)

    # Masking
    mask = torch.zeros(size=padded_sequences.shape[:2], dtype=torch.bool)
    for(i, s) in enumerate(sequences):
        mask[i, :len(s)] = True

    # Labeling
    labels = torch.tensor([int(l) for l in labels])

    return padded_sequences, mask, labels

In [32]:
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=32, collate_fn=collate_fn)

input_size = 126
d_model = 132

num_classes = len(label_id_dict)

model = LitKeypointTransformer(input_size=input_size, d_model=d_model, num_classes=num_classes)

early_stopping = EarlyStopping(monitor="val_loss", patience=3, mode="min")

trainer = L.Trainer(
    max_epochs=60,
    accelerator="auto",
    devices=1,
    callbacks=[early_stopping]
)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [33]:
x_batch, mask, y_batch = next(iter(train_loader))
logits = model(x_batch)  
print("x:", x_batch.shape, "y:", y_batch.shape, "logits:", logits.shape)

x: torch.Size([32, 538, 126]) y: torch.Size([32]) logits: torch.Size([32, 14])


In [34]:
trainer.fit(model, train_loader, val_loader)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name    | Type                | Params | Mode  | FLOPs
----------------------------------------------------------------
0 | model   | KeypointTransformer | 1.3 M  | train | 0    
1 | loss_fn | CrossEntropyLoss    | 0      | train | 0    
----------------------------------------------------------------
1.3 M     Trainable params
0         Non-trainable params
1.3 M     Total params
5.033     Total estimated model params size (MB)
91        Modules in train mode
0         Modules in eval mode
0         Total Flops


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/mark-oliver/git/gesture_recognition/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/mark-oliver/git/gesture_recognition/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.
/home/mark-oliver/git/gesture_recognition/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

In [35]:
from pathlib import Path
save_direct = Path("checkpoints")
trainer.save_checkpoint(save_direct / "keypoint_transformer_extensive.ckpt") 

`weights_only` was not set, defaulting to `False`.


In [36]:
model.eval()

LitKeypointTransformer(
  (model): KeypointTransformer(
    (input_proj): Sequential(
      (0): Linear(in_features=126, out_features=132, bias=True)
    )
    (pos_encode): SinusoidalPositionalEncoding()
    (layers): ModuleList(
      (0-5): 6 x EncodingLayer(
        (attention): MultiHeadAttention(
          (w_q): Linear(in_features=132, out_features=132, bias=True)
          (w_k): Linear(in_features=132, out_features=132, bias=True)
          (w_v): Linear(in_features=132, out_features=132, bias=True)
          (output): Linear(in_features=132, out_features=132, bias=True)
        )
        (norm1): LayerNorm((132,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((132,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
        (ffn): Sequential(
          (0): Linear(in_features=132, out_features=512, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_f

In [ ]:
#Testing model

In [37]:
all_labels = []
all_preds = []

for i in tqdm(range(len(test_ds))):    
    keypoint_sequence, label = test_ds[i]

    x = torch.tensor(keypoint_sequence, dtype=torch.float32).unsqueeze(0)  # (1, seq_len, 126)

    # prediction
    with torch.no_grad():
        logits = model(x)
        predicted_idx = torch.argmax(logits, dim=-1).item()

    all_labels.append(label)
    all_preds.append(predicted_idx)


  0%|          | 0/564 [00:00<?, ?it/s]

In [38]:
# F1 score and accuracy (again)
accuracy = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds, average="macro")
class_report = classification_report(all_labels, all_preds)

print(accuracy)
print(f1)
print(class_report)

0.8191489361702128
0.732842189896462
              precision    recall  f1-score   support

           0       0.87      0.92      0.89       148
           1       0.88      0.95      0.91       101
           2       0.94      0.85      0.89        98
           3       0.55      0.32      0.40        19
           4       0.83      0.50      0.62        20
           5       0.64      0.80      0.71        20
           6       0.78      0.70      0.74        20
           7       0.87      0.68      0.76        19
           8       0.84      0.80      0.82        20
           9       0.60      0.75      0.67        20
          10       0.46      0.65      0.54        20
          11       0.59      0.65      0.62        20
          12       1.00      0.85      0.92        20
          13       0.78      0.74      0.76        19

    accuracy                           0.82       564
   macro avg       0.76      0.73      0.73       564
weighted avg       0.83      0.82      0.82